In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

In [2]:
df = pd.read_csv('tempdataset.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
01-01-2010    27.483571
02-01-2010    24.308678
03-01-2010    28.238443
04-01-2010    32.615149
05-01-2010    23.829233


In [3]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df.values)

In [4]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

In [6]:
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - loss: 0.0217
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 14s 57ms/step - loss: 0.0181
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 13s 53ms/step - loss: 0.0179
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - loss: 0.0178
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - loss: 0.0181
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - loss: 0.0178
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - loss: 0.0178
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - loss: 0.0177
Epoch 9/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - loss: 0.0178
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - loss: 0.0177


In [7]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

predicted_values = scaler.inverse_transform(predicted_values)
print(
    f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
The predicted temperature for the next day is: 25.45°C
